# KOSIS 뉴스 사실검증 PoC 최소 스키마 구현

이 노트북은 최소 핵심 테이블 7개와 원본 API 보존 테이블을 SQLite로 구현한다.

1. `kosis_tables` — 통계표 카탈로그
2. `kosis_dimensions` — 통계표의 분류 축
3. `kosis_dimension_values` — 분류값
4. `kosis_items` — 검증 대상 지표·항목·단위
5. `kosis_periods` — 수록주기와 범위
6. `claims` — 구조화된 뉴스 주장
7. `verification_results` — 공식 통계 비교 결과
8. `raw_api_responses` — 재현성과 감사 목적의 원본 응답(보조)

생성 DB: `output/kosis_poc.db`

In [ ]:
import hashlib, json, os, sqlite3
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH = OUTPUT_DIR / 'kosis_poc.db'
ENV_PATH = ROOT / '.env'

def load_env(path=ENV_PATH):
    for raw in path.read_text(encoding='utf-8-sig').splitlines():
        line = raw.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            value = value.strip().strip(chr(34)).strip(chr(39))
            os.environ.setdefault(key.strip(), value)

load_env()
API_KEY = os.getenv('KOSIS_API_KEY', '').strip()
assert API_KEY, '.env에 KOSIS_API_KEY가 필요합니다.'
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')
print(DB_PATH.resolve())

## 1. 스키마 생성

KOSIS 식별자는 `org_id + tbl_id` 조합을 기준으로 한다. API 응답의 배열·별칭·차원조건은 SQLite JSON 문자열로 저장한다. 운영 DB가 PostgreSQL이면 `TEXT` JSON 필드를 `JSONB`로 변경할 수 있다.

In [ ]:
SCHEMA_SQL = '''
CREATE TABLE IF NOT EXISTS kosis_tables (
    table_key TEXT PRIMARY KEY,
    org_id TEXT NOT NULL,
    tbl_id TEXT NOT NULL,
    tbl_name TEXT NOT NULL,
    tbl_name_eng TEXT,
    stat_id TEXT,
    stat_name TEXT,
    view_code TEXT,
    category_path TEXT,
    recommended INTEGER NOT NULL DEFAULT 0 CHECK (recommended IN (0, 1)),
    source_updated_at TEXT,
    retrieved_at TEXT NOT NULL,
    UNIQUE (org_id, tbl_id)
);

CREATE TABLE IF NOT EXISTS kosis_dimensions (
    table_key TEXT NOT NULL,
    dimension_id TEXT NOT NULL,
    dimension_name TEXT NOT NULL,
    dimension_name_eng TEXT,
    dimension_order INTEGER,
    dimension_type TEXT NOT NULL DEFAULT 'other' CHECK (dimension_type IN
        ('region','sex','age','population','industry','occupation','household','item','time','other')),
    required INTEGER NOT NULL DEFAULT 0 CHECK (required IN (0, 1)),
    PRIMARY KEY (table_key, dimension_id),
    FOREIGN KEY (table_key) REFERENCES kosis_tables(table_key) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS kosis_dimension_values (
    table_key TEXT NOT NULL,
    dimension_id TEXT NOT NULL,
    value_id TEXT NOT NULL,
    value_name TEXT NOT NULL,
    value_name_eng TEXT,
    parent_value_id TEXT,
    normalized_name TEXT,
    aliases_json TEXT NOT NULL DEFAULT '[]' CHECK (json_valid(aliases_json)),
    PRIMARY KEY (table_key, dimension_id, value_id),
    FOREIGN KEY (table_key, dimension_id)
        REFERENCES kosis_dimensions(table_key, dimension_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS kosis_items (
    table_key TEXT NOT NULL,
    item_id TEXT NOT NULL,
    item_name TEXT NOT NULL,
    item_name_eng TEXT,
    unit_id TEXT,
    unit_name TEXT,
    metric_concept TEXT,
    measure_type TEXT NOT NULL DEFAULT 'count' CHECK (measure_type IN
        ('count','rate','percentage','percentage_point','average','index','amount','change_rate','other')),
    normalized_unit TEXT,
    aliases_json TEXT NOT NULL DEFAULT '[]' CHECK (json_valid(aliases_json)),
    claim_template TEXT,
    PRIMARY KEY (table_key, item_id),
    FOREIGN KEY (table_key) REFERENCES kosis_tables(table_key) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS kosis_periods (
    table_key TEXT NOT NULL,
    period_type TEXT NOT NULL,
    period_name TEXT NOT NULL,
    start_period TEXT,
    end_period TEXT,
    period_format TEXT,
    reference_type TEXT,
    publication_lag_days INTEGER,
    PRIMARY KEY (table_key, period_type),
    FOREIGN KEY (table_key) REFERENCES kosis_tables(table_key) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS claims (
    claim_id TEXT PRIMARY KEY,
    article_id TEXT NOT NULL,
    claim_text TEXT NOT NULL,
    metric TEXT NOT NULL,
    claim_value REAL,
    claim_unit TEXT,
    normalized_value REAL,
    normalized_unit TEXT,
    time_expression TEXT,
    resolved_period TEXT,
    period_type TEXT,
    region TEXT,
    population TEXT,
    comparison_type TEXT NOT NULL DEFAULT 'point_value' CHECK (comparison_type IN
        ('point_value','change_amount','change_rate','ranking','maximum','minimum','share','trend')),
    verifiability TEXT NOT NULL DEFAULT 'candidate' CHECK (verifiability IN
        ('candidate','verifiable','not_verifiable','needs_review')),
    extracted_at TEXT NOT NULL,
    extraction_json TEXT NOT NULL DEFAULT '{}' CHECK (json_valid(extraction_json))
);

CREATE TABLE IF NOT EXISTS verification_results (
    verification_id TEXT PRIMARY KEY,
    claim_id TEXT NOT NULL,
    table_key TEXT,
    item_id TEXT,
    official_value REAL,
    official_unit TEXT,
    official_period TEXT,
    dimensions_json TEXT NOT NULL DEFAULT '{}' CHECK (json_valid(dimensions_json)),
    absolute_difference REAL,
    relative_difference REAL,
    verdict TEXT NOT NULL CHECK (verdict IN
        ('MATCH','MOSTLY_MATCH','MISMATCH','MISSING_CONTEXT','WRONG_COMPARISON',
         'STATISTICS_NOT_FOUND','INSUFFICIENT_INFORMATION','NEEDS_REVIEW')),
    reason_code TEXT,
    explanation TEXT,
    evidence_json TEXT NOT NULL DEFAULT '{}' CHECK (json_valid(evidence_json)),
    review_status TEXT NOT NULL DEFAULT 'pending' CHECK (review_status IN ('pending','approved','rejected')),
    verified_at TEXT NOT NULL,
    FOREIGN KEY (claim_id) REFERENCES claims(claim_id) ON DELETE CASCADE,
    FOREIGN KEY (table_key, item_id) REFERENCES kosis_items(table_key, item_id)
);

CREATE TABLE IF NOT EXISTS raw_api_responses (
    response_id TEXT PRIMARY KEY,
    endpoint TEXT NOT NULL,
    request_params_json TEXT NOT NULL CHECK (json_valid(request_params_json)),
    response_json TEXT NOT NULL CHECK (json_valid(response_json)),
    retrieved_at TEXT NOT NULL
);

CREATE INDEX IF NOT EXISTS idx_tables_name ON kosis_tables(tbl_name);
CREATE INDEX IF NOT EXISTS idx_dimension_values_name ON kosis_dimension_values(normalized_name);
CREATE INDEX IF NOT EXISTS idx_items_name ON kosis_items(item_name, metric_concept);
CREATE INDEX IF NOT EXISTS idx_claims_metric_period ON claims(metric, resolved_period);
CREATE INDEX IF NOT EXISTS idx_results_claim ON verification_results(claim_id);
'''
conn.executescript(SCHEMA_SQL)
conn.commit()
print('스키마 생성 완료')

In [ ]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
tables

## 2. KOSIS 메타데이터 조회 및 정규화

`TBL`, `PRD`, `ITM` 메타데이터를 호출한다. `ITM` 응답에는 지표 항목(`OBJ_ID=ITEM`)과 분류값이 함께 있으므로 분리하여 적재한다.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def table_key(org_id, tbl_id):
    return f'{org_id}:{tbl_id}'

def kosis_meta(org_id, tbl_id, meta_type):
    endpoint = 'https://kosis.kr/openapi/statisticsData.do'
    public_params = {'method':'getMeta','type':meta_type,'orgId':org_id,'tblId':tbl_id,'format':'json','jsonVD':'Y'}
    request_params = {**public_params, 'apiKey': API_KEY}
    req = Request(f'{endpoint}?{urlencode(request_params)}', headers={'User-Agent':'kosis-schema-poc/1.0'})
    with urlopen(req, timeout=40) as response:
        payload = json.loads(response.read().decode('utf-8'))
    retrieved_at = utc_now()
    digest = hashlib.sha256((endpoint + json.dumps(public_params, sort_keys=True) + retrieved_at).encode()).hexdigest()
    conn.execute(
        'INSERT OR REPLACE INTO raw_api_responses VALUES (?, ?, ?, ?, ?)',
        (digest, endpoint, json.dumps(public_params, ensure_ascii=False),
         json.dumps(payload, ensure_ascii=False), retrieved_at)
    )
    return payload

PERIOD_CODES = {'일':'D','월':'M','격월':'M2','분기':'Q','반기':'S','년':'Y',
                '2년':'F2','3년':'F3','4년':'F4','5년':'F5','10년':'F10','부정기':'IR'}
PERIOD_FORMATS = {'D':'YYYYMMDD','M':'YYYYMM','M2':'YYYYMM','Q':'YYYYQQ',
                  'S':'YYYYHH','Y':'YYYY','F2':'YYYY','F3':'YYYY','F4':'YYYY',
                  'F5':'YYYY','F10':'YYYY','IR':'variable'}

def infer_dimension_type(name):
    rules = [('지역','region'),('행정구역','region'),('성별','sex'),('연령','age'),
             ('산업','industry'),('직업','occupation'),('가구','household'),('항목','item')]
    return next((kind for token, kind in rules if token in str(name)), 'other')

def infer_measure_type(item_name, unit_name):
    text = f'{item_name} {unit_name}'
    if '%p' in text: return 'percentage_point'
    if '%' in text or '비율' in text: return 'percentage'
    if '증감률' in text or '증가율' in text: return 'change_rate'
    if '지수' in text: return 'index'
    if '평균' in text: return 'average'
    if any(x in str(unit_name) for x in ['원','달러']): return 'amount'
    return 'count'

def normalize_unit(unit):
    return {'명':'person','천명':'person','만명':'person','세대':'household',
            '가구':'household','%':'percent','%p':'percentage_point',
            '원':'krw','만원':'krw','억원':'krw'}.get(unit, unit)


In [ ]:
def upsert_kosis_table(org_id, tbl_id, catalog=None):
    catalog = catalog or {}
    key = table_key(org_id, tbl_id)
    tbl_meta = kosis_meta(org_id, tbl_id, 'TBL')
    prd_meta = kosis_meta(org_id, tbl_id, 'PRD')
    itm_meta = kosis_meta(org_id, tbl_id, 'ITM')
    title = tbl_meta[0] if tbl_meta else {}
    now = utc_now()

    with conn:
        conn.execute('''
            INSERT INTO kosis_tables
            (table_key, org_id, tbl_id, tbl_name, tbl_name_eng, stat_id, stat_name,
             view_code, category_path, recommended, source_updated_at, retrieved_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(table_key) DO UPDATE SET
              tbl_name=excluded.tbl_name, tbl_name_eng=excluded.tbl_name_eng,
              source_updated_at=excluded.source_updated_at, retrieved_at=excluded.retrieved_at
        ''', (key, org_id, tbl_id, title.get('TBL_NM', catalog.get('TBL_NM', tbl_id)),
              title.get('TBL_NM_ENG'), catalog.get('STAT_ID'), catalog.get('STAT_NM'),
              catalog.get('VW_CD'), catalog.get('CATEGORY_PATH'),
              1 if catalog.get('REC_TBL_SE') == 'Y' else 0, catalog.get('SEND_DE'), now))

        for p in prd_meta:
            name = p.get('PRD_SE', '부정기')
            code = PERIOD_CODES.get(name, name)
            conn.execute('''
                INSERT OR REPLACE INTO kosis_periods
                (table_key, period_type, period_name, start_period, end_period, period_format)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (key, code, name, p.get('STRT_PRD_DE'), p.get('END_PRD_DE'), PERIOD_FORMATS.get(code)))

        dimensions = {}
        for row in itm_meta:
            dim_id, dim_name = row.get('OBJ_ID'), row.get('OBJ_NM')
            if dim_id and dim_name:
                dimensions[dim_id] = row
        for dim_id, row in dimensions.items():
            conn.execute('''
                INSERT OR REPLACE INTO kosis_dimensions
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (key, dim_id, row.get('OBJ_NM'), row.get('OBJ_NM_ENG'),
                  int(row['OBJ_ID_SN']) if str(row.get('OBJ_ID_SN','')).isdigit() else None,
                  infer_dimension_type(row.get('OBJ_NM')), 1))

        for row in itm_meta:
            dim_id, value_id = row.get('OBJ_ID'), row.get('ITM_ID')
            if not dim_id or not value_id:
                continue
            if dim_id == 'ITEM':
                conn.execute('''
                    INSERT OR REPLACE INTO kosis_items
                    (table_key, item_id, item_name, item_name_eng, unit_id, unit_name,
                     metric_concept, measure_type, normalized_unit, aliases_json, claim_template)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''', (key, value_id, row.get('ITM_NM'), row.get('ITM_NM_ENG'),
                      row.get('UNIT_ID'), row.get('UNIT_NM'), row.get('ITM_NM'),
                      infer_measure_type(row.get('ITM_NM'), row.get('UNIT_NM')),
                      normalize_unit(row.get('UNIT_NM')), '[]',
                      '{지역}의 {시점} ' + str(row.get('ITM_NM')) + '는 {값}' + str(row.get('UNIT_NM') or '') + '이다'))
            else:
                conn.execute('''
                    INSERT OR REPLACE INTO kosis_dimension_values
                    (table_key, dimension_id, value_id, value_name, value_name_eng,
                     parent_value_id, normalized_name, aliases_json)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ''', (key, dim_id, value_id, row.get('ITM_NM'), row.get('ITM_NM_ENG'),
                      row.get('UP_ITM_ID'), row.get('ITM_NM'), '[]'))
    return key


## 3. 주민등록인구 통계표 적재

샘플은 실제 검증 후보로 확인한 `DT_1B040A3`을 사용한다. 동일 함수를 다른 `org_id`, `tbl_id`에 반복 적용할 수 있다.

In [ ]:
resident_table_key = upsert_kosis_table(
    '101', 'DT_1B040A3',
    {'STAT_NM':'주민등록인구현황', 'VW_CD':'MT_ZTITLE',
     'CATEGORY_PATH':'인구 > 주민등록인구현황'}
)
print('적재 완료:', resident_table_key)

In [ ]:
summary = pd.read_sql_query('''
SELECT t.table_key, t.tbl_name,
       COUNT(DISTINCT d.dimension_id) AS dimension_count,
       COUNT(DISTINCT v.dimension_id || ':' || v.value_id) AS dimension_value_count,
       COUNT(DISTINCT i.item_id) AS item_count,
       GROUP_CONCAT(DISTINCT i.unit_name) AS units,
       GROUP_CONCAT(DISTINCT p.period_name) AS periods
FROM kosis_tables t
LEFT JOIN kosis_dimensions d ON d.table_key=t.table_key
LEFT JOIN kosis_dimension_values v ON v.table_key=t.table_key
LEFT JOIN kosis_items i ON i.table_key=t.table_key
LEFT JOIN kosis_periods p ON p.table_key=t.table_key
GROUP BY t.table_key, t.tbl_name
''', conn)
summary

## 4. 뉴스 주장과 검증 결과 저장 예시

이 셀은 스키마 연결을 검증하기 위한 예시다. 공식 수치는 실제 데이터 API 조회 결과로 교체해야 하며, 예시값을 최종 판정에 사용하면 안 된다.

In [ ]:
claim_id = 'demo-claim-001'
with conn:
    conn.execute('''
        INSERT OR REPLACE INTO claims
        (claim_id, article_id, claim_text, metric, claim_value, claim_unit,
         normalized_value, normalized_unit, time_expression, resolved_period, period_type,
         region, population, comparison_type, verifiability, extracted_at, extraction_json)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (claim_id, 'demo-article-001', '지난해 서울 인구는 950만 명이었다.',
          '주민등록인구', 950, '만명', 9_500_000, 'person', '지난해', '2025', 'Y',
          '서울', '전체', 'point_value', 'verifiable', utc_now(),
          json.dumps({'time_basis':'기사 발행일 기준'}, ensure_ascii=False)))

# 실제 공식값 조회 전이므로 NEEDS_REVIEW로 저장한다.
    conn.execute('''
        INSERT OR REPLACE INTO verification_results
        (verification_id, claim_id, table_key, item_id, official_value, official_unit,
         official_period, dimensions_json, verdict, reason_code, explanation,
         evidence_json, review_status, verified_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', ('demo-verification-001', claim_id, resident_table_key, 'T20', None, 'person',
          '2025', json.dumps({'region':'서울'}, ensure_ascii=False), 'NEEDS_REVIEW',
          'OFFICIAL_VALUE_NOT_FETCHED', '통계표와 항목은 찾았으나 공식 수치 조회 전입니다.',
          json.dumps({'org_id':'101','tbl_id':'DT_1B040A3','item_id':'T20'}, ensure_ascii=False),
          'pending', utc_now()))

pd.read_sql_query('''
SELECT c.claim_text, c.normalized_value AS claim_value, c.normalized_unit,
       t.tbl_name, i.item_name, v.verdict, v.reason_code
FROM verification_results v
JOIN claims c ON c.claim_id=v.claim_id
LEFT JOIN kosis_tables t ON t.table_key=v.table_key
LEFT JOIN kosis_items i ON i.table_key=v.table_key AND i.item_id=v.item_id
''', conn)

## 5. 무결성 및 완료 기준 확인

외래키 오류가 없어야 하며, 모든 KOSIS 표에는 항목과 수록주기가 존재해야 한다. 분류값이 없는 표는 구조상 가능하므로 경고 대상으로만 본다.

In [ ]:
foreign_key_errors = conn.execute('PRAGMA foreign_key_check').fetchall()
assert not foreign_key_errors, foreign_key_errors
quality = pd.read_sql_query('''
SELECT t.table_key, t.tbl_name,
       EXISTS(SELECT 1 FROM kosis_items i WHERE i.table_key=t.table_key) AS has_items,
       EXISTS(SELECT 1 FROM kosis_periods p WHERE p.table_key=t.table_key) AS has_periods,
       EXISTS(SELECT 1 FROM kosis_dimensions d WHERE d.table_key=t.table_key) AS has_dimensions
FROM kosis_tables t
''', conn)
assert quality['has_items'].all(), '항목 없는 통계표가 있습니다.'
assert quality['has_periods'].all(), '수록주기 없는 통계표가 있습니다.'
print('외래키 및 필수 메타데이터 검증 통과')
quality

In [ ]:
# 데이터 사전 출력
dictionary_rows = []
for table in ['kosis_tables','kosis_dimensions','kosis_dimension_values','kosis_items',
              'kosis_periods','claims','verification_results','raw_api_responses']:
    for col in conn.execute(f'PRAGMA table_info({table})').fetchall():
        dictionary_rows.append({'table':table, 'column':col[1], 'type':col[2],
                                'not_null':bool(col[3]), 'primary_key':bool(col[5])})
data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(OUTPUT_DIR / 'kosis_schema_data_dictionary.csv', index=False, encoding='utf-8-sig')
print('DB:', DB_PATH.resolve())
print('데이터 사전:', (OUTPUT_DIR / 'kosis_schema_data_dictionary.csv').resolve())
data_dictionary

## 다음 확장 순서

최소 스키마 이후에는 `(1)` 주장-통계표 후보 매핑과 점수 테이블, `(2)` 실제 관측값 캐시, `(3)` 기사 원문 및 발행일 테이블 순으로 추가한다. 공식 수치 조회를 붙이기 전까지 검증 결과는 반드시 `NEEDS_REVIEW`로 유지한다.